# DDPG

In [1]:
from mountaincar_utils import test_car, env_mountaincar, display_frames_as_gif, ReplayMemory
from IPython.display import HTML

In [2]:
import torch
import torch.nn as nn

# networks

# Q value net
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, action_dim)

    def forward(self, state, action):
        out = torch.cat((state, action), dim=-1)
        out = torch.relu(self.fc1(out))
        return self.fc2(out)
    
# policy net
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, action_dim)

    def forward(self, state):
        out = torch.relu(self.fc1(state))
        logits = self.fc2(out)
        return torch.tanh(logits)

In [3]:
import numpy as np

class OrnsteinUhlenbeckActionNoise:
    def __init__(self, mu, sigma, theta=.15, dt=1e-2, x0=None):
        self.theta = theta
        self.mu = mu
        self.sigma = sigma
        self.dt = dt
        self.x0 = x0
        self.reset()

    def __call__(self):
        x = self.x_prev + self.theta * (self.mu - self.x_prev) * self.dt + self.sigma * np.sqrt(self.dt) * np.random.normal(size=self.mu.shape)
        self.x_prev = x
        return x

    def reset(self):
        self.x_prev = self.x0 if self.x0 is not None else np.zeros_like(self.mu)

In [14]:
import random
import torch.nn.functional as F

class DDPMAgent:
    def __init__(self):
        self.qvalues = QNet(3, 1)
        self.qtargets = QNet(3, 1).requires_grad_(False)
        self.pvalues = PolicyNet(2, 1)
        self.ptargets = PolicyNet(2, 1).requires_grad_(False)
        self.action_noise = OrnsteinUhlenbeckActionNoise(mu=np.zeros(1), sigma=np.ones(1) * 0.05)
        self.opt_qvalues = torch.optim.AdamW(self.qvalues.parameters(), lr=0.001)
        self.opt_policy = torch.optim.AdamW(self.pvalues.parameters(), lr=0.001)

        self.epsilon = 1.0 #exploration rate
        self.epsilon_decay = 1.0/(200000)
        self.epsilon_final = 0.1
        self.reset()

    def reset(self):
        self.action_value = 0.0
        self.action_delta = 0.0
        self.previous_action_value = 0.0
        self.action_noise.reset()
    

    def act(self, state, train=True):
        if np.random.random() < self.epsilon and train:
            self.action_delta = random.uniform(-1., 1.)
        else:
            if isinstance(state, np.ndarray):
                if len(state.shape) == 1:
                    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                else:
                    state = torch.tensor(state, dtype=torch.float32)
            with torch.no_grad():
                noise = self.action_noise()
                action = self.pvalues(state)
                if train:
                    action += torch.tensor(noise, dtype=torch.float32)

            self.previous_action_value = self.action_value
            
            self.action_delta = action.detach().clone().item()
            self.action_delta = np.clip(self.action_delta, -1.0, 1.0)
        self.action_value += self.action_delta
        self.action_value = np.clip(self.action_value, -1.0, 1.0)
        env_input = np.array([self.action_value], dtype=np.float32)
        return env_input
    
    def update_epsilon(self):
        self.epsilon -= self.epsilon_decay
        if self.epsilon < self.epsilon_final:
            self.epsilon = self.epsilon_final
    
    def learn(self, plays, batch_size):
        gamma = 0.99

        # get data
        samples = plays.sample(batch_size)
        for sample in samples:
            if len(sample.shape) == 1:
                sample.unsqueeze_(-1)
        
        # get samples batch
        states, rewards, actions, states_next, dones = samples

        # critic
        q_values = self.qvalues(states, actions)
        with torch.no_grad():
            theta_target = self.ptargets(states_next)

            q_next = self.qtargets(states_next, theta_target)

            q_targets = rewards + (1.0 - dones) * gamma * q_next

        q_loss = F.mse_loss(q_values, q_targets, reduction='mean')

        self.opt_qvalues.zero_grad()
        q_loss.backward()
        self.opt_qvalues.step()

        # actor
        theta = self.pvalues(states)
        self.qvalues.requires_grad_(False)
        q_target_max = self.qvalues(states, theta)

        policy_loss = -q_target_max.mean()

        self.opt_policy.zero_grad()
        policy_loss.backward()
        self.opt_policy.step()

        self.qvalues.requires_grad_(True)
        
        # update target networks
        self.update_target()
        self.update_epsilon()

        return q_loss.detach().item(), policy_loss.detach().item()
    
    def update_target(self):
        tau = 0.01
        for target, source in zip(self.qtargets.parameters(), self.qvalues.parameters()):
            target.data.copy_(tau * source.data + (1.0 - tau) * target.data)

        for target, source in zip(self.ptargets.parameters(), self.pvalues.parameters()):
            target.data.copy_(tau * source.data + (1.0 - tau) * target.data)

In [ ]:
# train loop
from collections import deque

epochs = 2000
batch_size = 128
state_num = 2 # 
action_num = 1
memory = ReplayMemory(1000, control=True)
agent = DDPMAgent()

scores = []
losses = []
recent_scores = deque(maxlen=100)

for e in range(epochs):
    # reset environment
    state, _ = env_mountaincar.reset()
    agent.reset()

    currState = state
    done = False

    score = 0
    tot_loss = [0., 0.]
    count = 0
    tot_reward = 0.0

    # run an episode
    while not done :
        
        # choose action
        action = agent.act(state)

        # take action on env
        state, reward, terminated, truncated, info = env_mountaincar.step(action)
        done = terminated or truncated
        if score > 800:
            reward = -100.0
            done = True
        
        # add to replay memory
        memory.add([currState, reward, agent.previous_action_value, state, (1.0 if done else 0.0)])

        if len(memory) >= batch_size:
            # train the network
            qloss, ploss = agent.learn(memory, batch_size)
            tot_loss[0] += qloss
            tot_loss[1] += ploss
            count += 1

        currState = state.copy()
        # update score
        score += 1
        tot_reward += reward

    scores = np.append(scores, score)
    tmp = [l/count if count > 0 else 0. for l in tot_loss]
    losses = np.append(losses, tmp)
    recent_scores.append(tot_reward)

    if (e+1)%100 == 0:
        print(f"epoche: {e+1}, reward: {tot_reward}, score: {score}, qloss: {tot_loss[0]/count:.4f}, ploss: {tot_loss[1]/count:.4f}, epsilon: {agent.epsilon:.4f}")
    # early stopping if the goal is reached
    if len(recent_scores) >= 100:
        average_reward = sum(recent_scores) / 100
        if average_reward > 80.:
            print(f"Early stopping at episode {e+1} with average reward: {average_reward:.2f}")
            break

epoche: 100, reward: -48.363106093323644, score: 801, qloss: 0.0256, ploss: -6.5301, epsilon: 0.7601
epoche: 200, reward: -66.31610605612383, score: 801, qloss: 0.0056, ploss: 6.2637, epsilon: 0.3908
Early stopping at episode 297 with average reward: 80.25


In [16]:
frames = test_car(env_mountaincar, 900, agent=agent)
anim = display_frames_as_gif(frames)
HTML(anim.to_jshtml())

84
